# Calculate ETCCDI climate indices — UKESM1-1

This notebook calculates annual ETCCDI climate indices for UKESM1-1 experiments.
Two different S3 data sources are used, each with different file layouts and variable naming:

| Scenario | Bucket | Style | Members |
|---|---|---|---|
| G6-1.5K-HiLLA | `reflective-persistent-prod-large` | CMIP6 netCDF, 2-file split, standard names | r12i1p1f2, r2i1p1f2, r3i1p1f2 |
| G6-1.5K-SAI | `reflective-persistent-prod/alistairduffey` | Single file per member, CESM-like names | 001, 002, 003 |
| SSP245 | `reflective-persistent-prod/alistairduffey` | Single file per member, CESM-like names | 001, 002, 003 |

SOURCE: https://xclim.readthedocs.io/en/stable/indices.html

In [1]:
#

In [2]:
import xarray as xr
import numpy as np
import gc
from os.path import join
import xclim.indices
import os
import s3fs
import fsspec
import dask

In [3]:
# -------------------------------------------------------
# Configuration
# -------------------------------------------------------

model = 'UKESM1-1'

# --- G6-1.5K-HiLLA: reflective-persistent-prod-large, CMIP6 style ---
HILLA_S3_BASE = 's3://reflective-persistent-prod-large/UKESM1-1/G6-1p5K-HiLLA/'
HILLA_MEMBERS = {
    'r12i1p1f2': 'r1',
    'r2i1p1f2':  'r2',
    'r3i1p1f2':  'r3',
}

# --- G6-1.5K-SAI and SSP245: alistairduffey bucket, CESM-like style ---
ALISTAIR_S3_BASE = 's3://reflective-persistent-prod/alistairduffey/UKESM1-1/'
ALISTAIR_SCENARIOS = ['G6-1.5K-SAI', 'SSP245']
ALISTAIR_MEMBERS = {'r1': '001', 'r2': '002', 'r3': '003'}

# S3 output root — same structure as the local out_data/ directory was
OUTPUT_ROOT = f's3://reflective-persistent-prod/alistairduffey/ETCCDI/ETCCDI_indices_annual/{model}'

fs = s3fs.S3FileSystem(anon=False)

def save_nc(obj, s3_path):
    """Write a DataArray or Dataset to S3 via a local temp file (h5netcdf engine)."""
    import tempfile
    with tempfile.NamedTemporaryFile(suffix='.nc', delete=False) as f:
        tmp = f.name
    try:
        obj.to_netcdf(tmp, engine='h5netcdf')
        fs.put(tmp, s3_path.replace('s3://', ''))
    finally:
        os.remove(tmp)

# UKESM1-1 n96e grid: 144 lat × 192 lon
CHUNKS = {'time': 365}

## Temperature indices

| Variable Used | Index | Name | Definition | Unit | Type |
|---|---|---|---|---|---|
| tasmax (Tx) | SU | Summer days | Number of days when Tx $>$ 25$^\circ$C | days per year | Fixed threshold |
| tasmax (Tx) | ID | Ice days | Number of days when Tx $<$ 0$^\circ$C | days per year | Fixed threshold |
| tasmax (Tx) | TXn | Coldest days | Annual minimum daily maximum temperature | $^\circ$C | Fixed Index |
| tasmax (Tx) | TXx | Warmest days | Annual maximum daily maximum temperature | $^\circ$C | Fixed Index |
| tasmin (Tn) | TNn | Coldest night | Annual minimum daily minimum temperature | $^\circ$C | Fixed Index |
| tasmin (Tn) | TNx | Warmest night | Annual maximum daily minimum temperature | days per year | Fixed Index |
| tasmin (Tn) | FD | Frost days | Number of days when Tn < 0$^\circ$C | days per year | Fixed threshold |
| tasmin (Tn) | TN | Tropical nights | Number of days when Tn $>$ 20$^\circ$C | days per year | Fixed threshold |

### G6-1.5K-HiLLA (CMIP6 style, reflective-persistent-prod-large)

In [4]:
for member_full, member_short in HILLA_MEMBERS.items():
    print(f'Processing scenario=G6-1.5K-HiLLA, member={member_short} ({member_full})')

    base = f'{HILLA_S3_BASE}{member_full}/'

    tasmax_files = sorted(fs.glob(f'{base}apd/day/tasmax/*.nc'))
    tasmin_files = sorted(fs.glob(f'{base}apd/day/tasmin/*.nc'))

    tasmax_ds = xr.open_mfdataset(
        [fsspec.open(f's3://{f}', mode='rb', anon=False).open() for f in tasmax_files],
        combine='nested', concat_dim='time',
        engine='h5netcdf', chunks=CHUNKS,
        coords='minimal', compat='override'
    )
    tasmin_ds = xr.open_mfdataset(
        [fsspec.open(f's3://{f}', mode='rb', anon=False).open() for f in tasmin_files],
        combine='nested', concat_dim='time',
        engine='h5netcdf', chunks=CHUNKS,
        coords='minimal', compat='override'
    )

    tasmax_ds = tasmax_ds.sel(time=~tasmax_ds.get_index('time').duplicated())
    tasmin_ds = tasmin_ds.sel(time=~tasmin_ds.get_index('time').duplicated())
    tasmax_ds = tasmax_ds.sel(time=(tasmax_ds.time.dt.year >= 2035) & (tasmax_ds.time.dt.year <= 2084))
    tasmin_ds = tasmin_ds.sel(time=(tasmin_ds.time.dt.year >= 2035) & (tasmin_ds.time.dt.year <= 2084))

    tx = tasmax_ds['tasmax']
    tn = tasmin_ds['tasmin']

    SU  = xclim.indices.tx_days_above(tx, thresh='25.0 degC', freq='YS')
    ID  = xclim.indices.ice_days(tx, thresh='0 degC', freq='YS')
    TXX = xclim.indices.tx_max(tx, freq='YS')
    TXN = xclim.indices.tx_min(tx, freq='YS')
    FD  = xclim.indices.frost_days(tn, freq='YS')
    TN  = xclim.indices.tn_days_above(tn, thresh='20.0 degC', freq='YS')
    TNX = xclim.indices.tn_max(tn, freq='YS')
    TNN = xclim.indices.tn_min(tn, freq='YS')

    su, id_, txx, txn, fd, tn_, tnx, tnn = dask.compute(SU, ID, TXX, TXN, FD, TN, TNX, TNN)

    out_dir = join(OUTPUT_ROOT, 'G6-1.5K-HiLLA', member_short)

    save_nc(su,  join(out_dir, 'SU.nc'))
    save_nc(id_, join(out_dir, 'ID.nc'))
    save_nc(txx, join(out_dir, 'TXX.nc'))
    save_nc(txn, join(out_dir, 'TXN.nc'))
    save_nc(fd,  join(out_dir, 'FD.nc'))
    save_nc(tn_, join(out_dir, 'TN.nc'))
    save_nc(tnx, join(out_dir, 'TNX.nc'))
    save_nc(tnn, join(out_dir, 'TNN.nc'))

    del tasmax_ds, tasmin_ds, tx, tn, su, id_, txx, txn, fd, tn_, tnx, tnn
    gc.collect()
    print(f'  -> Temperature indices saved to {out_dir}')

Processing scenario=G6-1.5K-HiLLA, member=r1 (r12i1p1f2)
  -> Temperature indices saved to s3://reflective-persistent-prod/alistairduffey/ETCCDI/ETCCDI_indices_annual/UKESM1-1/G6-1.5K-HiLLA/r1
Processing scenario=G6-1.5K-HiLLA, member=r2 (r2i1p1f2)
  -> Temperature indices saved to s3://reflective-persistent-prod/alistairduffey/ETCCDI/ETCCDI_indices_annual/UKESM1-1/G6-1.5K-HiLLA/r2
Processing scenario=G6-1.5K-HiLLA, member=r3 (r3i1p1f2)
  -> Temperature indices saved to s3://reflective-persistent-prod/alistairduffey/ETCCDI/ETCCDI_indices_annual/UKESM1-1/G6-1.5K-HiLLA/r3


### G6-1.5K-SAI and SSP245 (alistairduffey bucket)

Temperature data is in a single file per member: `daily_Tmaxmin/T_{num}.nc`  
Variable mapping: `temp` = tasmax, `temp_1` = tasmin

In [ ]:
for scenario in ALISTAIR_SCENARIOS:
    for member, member_num in ALISTAIR_MEMBERS.items():
        print(f'Processing scenario={scenario}, member={member}')

        t_path = f'{ALISTAIR_S3_BASE}{scenario}/daily_Tmaxmin/T_{member_num}.nc'

        ds = xr.open_dataset(
            fsspec.open(t_path, mode='rb', anon=False).open(),
            engine='scipy', chunks={'t': 365}
        )
        ds = ds.rename({'t': 'time'}).squeeze('ht', drop=True)

        # Limit to 2035–2084 for G6 scenarios only; use full range for SSP245
        if scenario != 'SSP245':
            ds = ds.sel(time=(ds.time.dt.year >= 2035) & (ds.time.dt.year <= 2084))

        tx = ds['temp']
        tn = ds['temp_1']

        out_dir = join(OUTPUT_ROOT, scenario, member)

        # --- Pass 1: tasmax-based indices ---
        su, id_, txx, txn = dask.compute(
            xclim.indices.tx_days_above(tx, thresh='25.0 degC', freq='YS'),
            xclim.indices.ice_days(tx, thresh='0 degC', freq='YS'),
            xclim.indices.tx_max(tx, freq='YS'),
            xclim.indices.tx_min(tx, freq='YS'),
        )
        save_nc(su,  join(out_dir, 'SU.nc'));  del su
        save_nc(id_, join(out_dir, 'ID.nc'));  del id_
        save_nc(txx, join(out_dir, 'TXX.nc')); del txx
        save_nc(txn, join(out_dir, 'TXN.nc')); del txn
        gc.collect()

        # --- Pass 2: tasmin-based indices ---
        fd, tn_, tnx, tnn = dask.compute(
            xclim.indices.frost_days(tn, thresh='0 degC', freq='YS'),
            xclim.indices.tn_days_above(tn, thresh='20.0 degC', freq='YS'),
            xclim.indices.tn_max(tn, freq='YS'),
            xclim.indices.tn_min(tn, freq='YS'),
        )
        save_nc(fd,  join(out_dir, 'FD.nc'));   del fd
        save_nc(tn_, join(out_dir, 'TN.nc'));   del tn_
        save_nc(tnx, join(out_dir, 'TNX.nc'));  del tnx
        save_nc(tnn, join(out_dir, 'TNN.nc'));  del tnn

        del ds, tx, tn
        gc.collect()
        print(f'  -> Temperature indices saved to {out_dir}')

## Precipitation indices

In [ ]:
# To convert precipitation rate to mm/day
SECONDS_PER_DAY = 86400
M_TO_MM = 1000

def to_mm_day(da, varname=None):
    """
    Convert precipitation rate to mm/day when units look like:
      - kg m-2 s-1  (water flux; 1 kg/m^2 = 1 mm water)
      - m s-1       (water-equivalent depth rate)
    Otherwise return da unchanged.
    """
    units = (da.attrs.get("units") or "").strip().lower().replace(" ", "")

    is_kg_m2_s = units in {"kgm-2s-1", "kg/m^2/s", "kg/m2/s", "kgm^-2s^-1", "kgm**-2s**-1"}
    is_m_s     = units in {"ms-1", "m/s", "m s-1", "m/s-1"} or units == "ms^-1"

    if is_kg_m2_s:
        out = da * SECONDS_PER_DAY
    elif is_m_s:
        out = da * SECONDS_PER_DAY * M_TO_MM
    else:
        if varname is not None and varname.upper() in {"PRECT", "PRECC", "PRECL", "PR"}:
            out = da * SECONDS_PER_DAY
        else:
            return da

    out = out.assign_attrs(da.attrs)
    out.attrs["units"] = "mm/day"
    return out

| Variable Used | Index | Name | Definition | Unit | Type |
|---|---|---|---|---|---|
| pr (PR) | PRCPTOT | Total rainfall | Annual sum of precipitation | mm | Fixed Index |
| pr (PR) | SDII | Simple daily intensity | Mean precipitation falling on days when PR $\geq$ 1 mm | mm | Fixed Index |
| pr (PR) | Rx1day | Wettest day | Annual maximum precipitation in a single day | mm | Fixed Index |
| pr (PR) | Rx5day | Wettest pentad | Annual maximum precipitation falling on 5 consecutive days | mm | Fixed Index |
| pr (PR) | CDD | Consecutive dry days | Longest spell of consecutive days when PR $\leq$ 1 mm | days per year | Fixed index/spell |
| pr (PR) | CWD | Consecutive wet days | Longest spell of consecutive days when PR $\geq$ 1 mm | days per year | Fixed index/spell |
| pr (PR) | R10mm | Heavy precipitation days | Number of days when precipitation $\geq$ 10 mm | days per year | Fixed threshold |
| pr (PR) | R20mm | Very heavy precipitation days | Number of days when precipitation $\geq$ 20 mm | days per year | Fixed threshold |

### G6-1.5K-HiLLA (CMIP6 style, reflective-persistent-prod-large)

In [ ]:
for member_full, member_short in HILLA_MEMBERS.items():
    print(f'Processing scenario=G6-1.5K-HiLLA, member={member_short} ({member_full})')

    pr_files = sorted(fs.glob(f'{HILLA_S3_BASE}{member_full}/apj/day/pr/*.nc'))

    ds = xr.open_mfdataset(
        [fsspec.open(f's3://{f}', mode='rb', anon=False).open() for f in pr_files],
        combine='nested', concat_dim='time',
        engine='h5netcdf', chunks=CHUNKS,
        coords='minimal', compat='override'
    )

    ds = ds.sel(time=~ds.get_index('time').duplicated())
    ds = ds.sel(time=(ds.time.dt.year >= 2035) & (ds.time.dt.year <= 2084))
    pr = to_mm_day(ds['pr'], varname='pr')

    out_dir = join(OUTPUT_ROOT, 'G6-1.5K-HiLLA', member_short)

    # --- Pass 1: simple per-timestep aggregations ---
    prcptot, sdii, rx1d, rx5d, r10mm, r20mm = dask.compute(
        xclim.indices.prcptot(pr, freq='YS'),
        xclim.indices.daily_pr_intensity(pr, thresh='1 mm/day', freq='YS'),
        xclim.indices.max_1day_precipitation_amount(pr, freq='YS'),
        xclim.indices.max_n_day_precipitation_amount(pr, window=5, freq='YS'),
        xclim.indices.wetdays(pr, thresh='10 mm/day', freq='YS', op='>='),
        xclim.indices.wetdays(pr, thresh='20 mm/day', freq='YS', op='>='),
    )
    save_nc(prcptot, join(out_dir, 'PRCPTOT.nc')); del prcptot
    save_nc(sdii,    join(out_dir, 'SDII.nc'));    del sdii
    save_nc(rx1d,    join(out_dir, 'RX1D.nc'));    del rx1d
    save_nc(rx5d,    join(out_dir, 'RX5D.nc'));    del rx5d
    save_nc(r10mm,   join(out_dir, 'R10MM.nc'));   del r10mm
    save_nc(r20mm,   join(out_dir, 'R20MM.nc'));   del r20mm
    gc.collect()

    # --- Pass 2: consecutive-spell indices ---
    cdd, cwd = dask.compute(
        xclim.indices.maximum_consecutive_dry_days(pr, '1 mm/day', freq='YS'),
        xclim.indices.maximum_consecutive_wet_days(pr, thresh='1 mm/day', freq='YS'),
    )
    save_nc(cdd, join(out_dir, 'CDD.nc')); del cdd
    save_nc(cwd, join(out_dir, 'CWD.nc')); del cwd

    del ds, pr
    gc.collect()
    print(f'  -> Precipitation indices saved to {out_dir}')

### G6-1.5K-SAI and SSP245 (alistairduffey bucket)

Precipitation data is in a single file per member: `daily_pr/PRECT_{num}.nc`  
Variable name: `PRECT` (units: m/s)

In [ ]:
for scenario in ALISTAIR_SCENARIOS:
    for member, member_num in ALISTAIR_MEMBERS.items():
        print(f'Processing scenario={scenario}, member={member}')

        pr_path = f'{ALISTAIR_S3_BASE}{scenario}/daily_pr/PRECT_{member_num}.nc'

        ds = xr.open_dataset(
            fsspec.open(pr_path, mode='rb', anon=False).open(),
            engine='scipy', chunks={'t': 365}
        )
        ds = ds.rename({'t': 'time', 'precip': 'PRECT'}).squeeze('surface', drop=True)

        # Limit to 2035–2084 for G6 scenarios only; use full range for SSP245
        if scenario != 'SSP245':
            ds = ds.sel(time=(ds.time.dt.year >= 2035) & (ds.time.dt.year <= 2084))

        pr = to_mm_day(ds['PRECT'], varname='PRECT')

        out_dir = join(OUTPUT_ROOT, scenario, member)

        # --- Pass 1: simple per-timestep aggregations ---
        prcptot, sdii, rx1d, rx5d, r10mm, r20mm = dask.compute(
            xclim.indices.prcptot(pr, freq='YS'),
            xclim.indices.daily_pr_intensity(pr, thresh='1 mm/day', freq='YS'),
            xclim.indices.max_1day_precipitation_amount(pr, freq='YS'),
            xclim.indices.max_n_day_precipitation_amount(pr, window=5, freq='YS'),
            xclim.indices.wetdays(pr, thresh='10 mm/day', freq='YS', op='>='),
            xclim.indices.wetdays(pr, thresh='20 mm/day', freq='YS', op='>='),
        )
        save_nc(prcptot, join(out_dir, 'PRCPTOT.nc')); del prcptot
        save_nc(sdii,    join(out_dir, 'SDII.nc'));    del sdii
        save_nc(rx1d,    join(out_dir, 'RX1D.nc'));    del rx1d
        save_nc(rx5d,    join(out_dir, 'RX5D.nc'));    del rx5d
        save_nc(r10mm,   join(out_dir, 'R10MM.nc'));   del r10mm
        save_nc(r20mm,   join(out_dir, 'R20MM.nc'));   del r20mm
        gc.collect()

        # --- Pass 2: consecutive-spell indices ---
        cdd, cwd = dask.compute(
            xclim.indices.maximum_consecutive_dry_days(pr, '1 mm/day', freq='YS'),
            xclim.indices.maximum_consecutive_wet_days(pr, thresh='1 mm/day', freq='YS'),
        )
        save_nc(cdd, join(out_dir, 'CDD.nc')); del cdd
        save_nc(cwd, join(out_dir, 'CWD.nc')); del cwd

        del ds, pr
        gc.collect()
        print(f'  -> Precipitation indices saved to {out_dir}')